# S3 Object Storage on FABRIC

FABRIC's Ceph clusters expose an **S3-compatible object store** (Ceph RGW) alongside
the POSIX CephFS filesystem shown in the other notebooks in this directory.

This notebook covers what FABlib does for S3:

1. Discover which clusters offer S3 and where their endpoints are
2. List the buckets you own
3. Fetch your S3 credentials and write ready-to-use client config
4. Use those credentials from a FABRIC node with `aws-cli`, `s3cmd`, or `boto3`
5. Measure the throughput you actually get

### Two things worth knowing up front

**FABlib does not move object data.** It is a control-plane helper: it finds
endpoints, lists buckets, and hands you credentials. Transfers are done with a
standard S3 client so that FABlib does not drag a `boto3` dependency into every
FABRIC install. Everything below works with any S3 tool you already use.

**Bucket creation and deletion are administrative.** Only FABRIC facility
administrators and owners of the *Service - FABRIC Ceph* project can create or
delete buckets. Request one through the **Storage → S3 Buckets** tab of the
Credential Manager portal. Once a bucket is yours, you have full read/write
access to its contents.

**Transfers run from a slice node, not from here.** RGW endpoints live on
FABNet, which JupyterHub and your laptop have no route to. The credentials are
fetched here and used on a node with a FABNet interface.

**Your S3 identity is your bastion login** — the same identity used to name your
CephFS subvolume.

## Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

## 1. Discover clusters and their S3 endpoints

Each FABRIC Ceph cluster is an independent RGW realm, so an S3 user and its keys
exist **per cluster** — the same uid on `east` and `west` are unrelated accounts
with different credentials.

In [ ]:
import json

clusters = fablib.discover_ceph_clusters()
print(f"Available storage clusters: {json.dumps(clusters, indent=2)}")

# Pick one to work with for the rest of the notebook.
CLUSTER = clusters[0] if clusters else "east"
print(f"\nUsing cluster: {CLUSTER}")

In [ ]:
from fabrictestbed_extensions.utils.ceph_s3_utils import CephS3Credentials

endpoints = CephS3Credentials.list_s3_endpoints(
    base_url=fablib.get_ceph_mgr_host(),
    cluster=CLUSTER,
    token_file=fablib.get_token_location(),
)
print(f"S3 endpoints for {CLUSTER}:")
for e in endpoints:
    print(f"  {e}")

print("\nNote: these are FABNet addresses. They are reachable from a FABRIC")
print("slice (any node with a FABNet interface), not from the public internet.")

## 2. List your buckets

This goes through the Ceph Manager API, so it needs no S3 client at all.

The service scopes the result by identity: unless you are an administrator you
will only ever see buckets you own, regardless of what you ask for.

In [ ]:
buckets = fablib.list_s3_buckets(cluster=CLUSTER)

if not buckets:
    print(f"No buckets found on {CLUSTER}.")
    print("Request one via the Credential Manager portal: Storage -> S3 Buckets.")
else:
    for b in buckets:
        print(f"  {b['name']:30} owner={b.get('owner'):20} "
              f"objects={b.get('num_objects')} versioning={b.get('versioning')}")

## 3. Get your S3 credentials

`get_s3_credentials()` returns an access key / secret key pair plus the endpoint
to point a client at.

An existing keypair is reused when one can be read back; a new one is minted only
if you have none. That matters — otherwise every call would leave another key
behind on your account.

> The secret is a long-lived credential. Treat the files below like an SSH private
> key: do not commit them, and do not paste them into a shared notebook output.

In [ ]:
creds = fablib.get_s3_credentials(cluster=CLUSTER)

print(f"cluster    : {creds['cluster']}")
print(f"uid        : {creds['uid']}")
print(f"endpoint   : {creds['endpoint']}")
print(f"access_key : {creds['access_key']}")
print(f"secret_key : {'*' * len(creds['secret_key'])}  ({len(creds['secret_key'])} chars)")

### Write ready-to-use client config

Passing `out_base` also writes config for the common S3 clients. Files holding a
secret are written mode `0600`.

In [ ]:
creds = fablib.get_s3_credentials(cluster=CLUSTER, out_base="./s3-artifacts")

for name, path in sorted(creds["files"].items()):
    print(f"  {name:16} {path}")

print("\n--- README.md ---")
print(open(creds["files"]["README.md"]).read())

## 4. Use S3 from a FABRIC node

This is the common case: a slice node reads and writes objects over FABNet.
The node needs a FABNet interface, which `add_fabnet()` provides.

In [ ]:
slice_name = "S3-Example"

slice1 = fablib.new_slice(name=slice_name)
node = slice1.add_node(name="s3-client", cores=2, ram=8, disk=50)
node.add_fabnet()          # required: RGW endpoints live on FABNet

slice1.submit();

In [ ]:
slice1 = fablib.get_slice(slice_name)
node = slice1.get_node(name="s3-client")

# Install a client. awscli is in the distro repos for Rocky/Ubuntu.
node.execute("sudo dnf install -y awscli || sudo apt-get install -y -qq awscli")

### Push the credentials to the node and run a round trip

In [ ]:
BUCKET = buckets[0]["name"] if buckets else "<your-bucket>"

node.execute("mkdir -p ~/.aws")
node.upload_file(creds["files"]["aws_credentials"], ".aws/credentials")
node.upload_file(creds["files"]["aws_config"], ".aws/config")
node.execute("chmod 600 ~/.aws/credentials")

profile = f"fabric-{CLUSTER}"
endpoint = creds["endpoint"]

# Round trip: write a file, upload, list, download, compare.
node.execute(f"""
set -e
echo 'hello from FABRIC' > /tmp/hello.txt
aws --profile {profile} --endpoint-url {endpoint} s3 cp /tmp/hello.txt s3://{BUCKET}/hello.txt
aws --profile {profile} --endpoint-url {endpoint} s3 ls s3://{BUCKET}/
aws --profile {profile} --endpoint-url {endpoint} s3 cp s3://{BUCKET}/hello.txt /tmp/hello-back.txt
diff /tmp/hello.txt /tmp/hello-back.txt && echo 'ROUND TRIP OK'
""")

### s3cmd

`write_client_config()` also emits an `.s3cfg` if you prefer `s3cmd`.

In [ ]:
# node.execute("sudo dnf install -y s3cmd || sudo apt-get install -y -qq s3cmd")
# node.upload_file(creds["files"]["s3cfg"], ".s3cfg")
# node.execute("chmod 600 ~/.s3cfg")
# node.execute(f"s3cmd ls s3://{BUCKET}/")
print("s3cmd snippet above is commented out.")

## 5. Measure throughput

How fast S3 actually is here depends on the object size, the concurrency of the
client, and the distance from your node to the RGW gateway. This section
measures all three so you can size a transfer realistically rather than guess.

**What is being measured.** Wall-clock time for `aws s3 cp` between the node's
local disk and RGW, over FABNet. That is end-to-end client throughput, so it
includes TLS, HTTP, RGW, and the replicated write down to the OSDs.

**Reading the numbers honestly:**

- Test files are generated with `/dev/urandom` *before* timing starts, so
  generation cost is excluded. Random data also avoids flattering any
  compression in the path.
- Downloads are read back into a fresh file, but nothing here drops RGW's or
  the OSDs' caches. A re-read of a just-written object can be served warm, so
  download figures are an optimistic bound.
- One sample per size. This shows you an order of magnitude, not a benchmark
  you should quote. Raise `REPEATS` if you want a number you can defend.
- Small objects are dominated by per-request overhead, not bandwidth, which is
  why they get their own test reported in objects/sec rather than MB/s.


In [ ]:
# Throughput test. Runs entirely on the node; nothing streams through this notebook.
SIZES_MB = [1, 10, 100, 1024]     # per-object sizes for the bandwidth test
SMALL_COUNT = 200                 # number of 64 KiB objects for the overhead test
REPEATS = 1                       # raise for a number worth quoting
PREFIX = "bench"                  # objects land under s3://$BUCKET/$PREFIX/

bench = f"""
set -eu
# A mid-run failure must not look like a short successful run: set -e stops the
# script, and node.execute() does not raise, so without this the cell below would
# tabulate whatever happened to print first.
trap 'rc=$?; echo "CSV,ABORTED,$rc,0"' ERR
PROFILE={profile}
EP={endpoint}
BUCKET={BUCKET}
WORK=/tmp/s3bench
AWS="aws --profile $PROFILE --endpoint-url $EP"

mkdir -p $WORK && cd $WORK

# Generate first, time later -- file creation is not part of the measurement.
for MB in {' '.join(str(s) for s in SIZES_MB)}; do
  [ -f obj_${{MB}}.bin ] || dd if=/dev/urandom of=obj_${{MB}}.bin bs=1M count=$MB status=none
done

# Nanosecond timers; `bc` is not installed everywhere, so use shell arithmetic.
for i in $(seq 1 {REPEATS}); do
  for MB in {' '.join(str(s) for s in SIZES_MB)}; do
    s=$(date +%s%N)
    $AWS s3 cp obj_${{MB}}.bin s3://$BUCKET/{PREFIX}/obj_${{MB}}.bin --only-show-errors
    e=$(date +%s%N)
    echo "CSV,upload,$MB,$(( (e - s) / 1000000 ))"

    rm -f dl_${{MB}}.bin
    s=$(date +%s%N)
    $AWS s3 cp s3://$BUCKET/{PREFIX}/obj_${{MB}}.bin dl_${{MB}}.bin --only-show-errors
    e=$(date +%s%N)
    echo "CSV,download,$MB,$(( (e - s) / 1000000 ))"

    cmp -s obj_${{MB}}.bin dl_${{MB}}.bin || echo "CSV,MISMATCH,$MB,0"
  done
done

# Many small objects: this measures request rate, not bandwidth.
rm -rf small && mkdir small
for n in $(seq 1 {SMALL_COUNT}); do dd if=/dev/urandom of=small/o_$n.bin bs=64K count=1 status=none; done
s=$(date +%s%N)
$AWS s3 sync small s3://$BUCKET/{PREFIX}/small/ --only-show-errors
e=$(date +%s%N)
echo "CSV,small_upload_{SMALL_COUNT}x64KiB,$(( {SMALL_COUNT} * 64 / 1024 )),$(( (e - s) / 1000000 ))"

echo "CSV,COMPLETE,0,0"
"""

stdout, stderr = node.execute(bench, quiet=True)
lines = [l for l in stdout.splitlines() if l.startswith("CSV,")]
tags = [l.split(",")[1] for l in lines]

# Fail loudly on a partial run. Anything short of the full set means a transfer
# died, and quietly charting the survivors would read as a complete result.
if "COMPLETE" not in tags:
    rc = next((l.split(",")[2] for l in lines if l.split(",")[1] == "ABORTED"), "?")
    raise RuntimeError(
        f"benchmark aborted (exit {rc}) after {len(lines)} of "
        f"{len(SIZES_MB) * REPEATS * 2 + 1} measurements.\n"
        f"stderr:\n{stderr.strip()[:1000] or '(empty)'}"
    )

raw = [l for l in lines if l.split(",")[1] not in ("COMPLETE", "ABORTED")]
expected = len(SIZES_MB) * REPEATS * 2 + 1          # up+down per size, plus the small-object run
got = len([l for l in raw if l.split(",")[1] != "MISMATCH"])
if got != expected:
    raise RuntimeError(f"expected {expected} measurements, got {got}: {raw}")

print(f"{got} measurements, run complete")
if stderr.strip():
    print("stderr:", stderr.strip()[:400])


In [ ]:
import pandas as pd

rows = []
for line in raw:
    _, op, mb, ms = line.split(",")
    mb, secs = float(mb), int(ms) / 1000.0
    if op == "MISMATCH":
        raise RuntimeError(f"downloaded object differed from the original at {mb} MiB")
    rows.append({
        "operation": op,
        "size (MiB)": mb,
        "seconds": round(secs, 2),
        "MiB/s": round(mb / secs, 1) if secs else float("nan"),
    })

df = pd.DataFrame(rows)

# Per-request cost is the interesting figure for the small-object run.
small = df[df["operation"].str.startswith("small_")]
if not small.empty:
    n = SMALL_COUNT
    secs = small.iloc[0]["seconds"]
    if secs > 0:
        print(f"small objects: {n} x 64 KiB in {secs:.1f}s "
              f"= {n / secs:.0f} objects/sec, {secs / n * 1000:.0f} ms per object")
    else:
        print(f"small objects: {n} x 64 KiB completed too quickly to time")

df


### What the shape of these numbers means

Expect throughput to climb steeply with object size and then flatten. The small
sizes are latency-bound — each `cp` is a fresh HTTP request, and at 1 MiB the
round trip costs more than the bytes do. Somewhere in the tens of MiB the
transfer starts to dominate and you approach what the path can carry.

If the large-object figure disappoints, the usual causes in order:

1. **Concurrency.** A single `aws s3 cp` uses a bounded worker pool. Raise it:
   `aws configure set default.s3.max_concurrent_requests 20`, and
   `default.s3.multipart_chunksize 16MB`.
2. **Distance.** Your node and the RGW gateway may be in different regions.
   `discover_ceph_clusters()` in section 1 shows where each cluster lives —
   picking the near one usually matters more than any client tuning.
3. **The node itself.** 2 cores is enough for `aws-cli`, but TLS on a small
   flavour can cap you before the network does. Compare against a larger node
   before blaming the storage.

Objects written by this section stay under `s3://$BUCKET/bench/`. Delete them
when you are done — they count against the bucket quota.


In [ ]:
# Remove the benchmark objects (they count against your quota).
node.execute(
    f"aws --profile {profile} --endpoint-url {endpoint} "
    f"s3 rm s3://{BUCKET}/{PREFIX}/ --recursive --only-show-errors && echo 'bench objects removed'"
)


## Cleanup

In [ ]:
slice1.delete()

## Reference

| What | How |
|---|---|
| Which clusters exist | `fablib.discover_ceph_clusters()` |
| S3 endpoints for a cluster | `CephS3Credentials.list_s3_endpoints(...)` |
| Buckets you own | `fablib.list_s3_buckets(cluster=...)` |
| Credentials + client config | `fablib.get_s3_credentials(cluster=..., out_base=...)` |
| Create / delete a bucket | Credential Manager portal → Storage → S3 Buckets (admins only) |
| Transfer objects | any S3 client, using the config written above |

**POSIX vs S3.** Use CephFS (the other notebooks here) when you want a shared
filesystem your jobs can `open()` and `mmap()`. Use S3 when you want
HTTP-accessible objects, versioning, or data shared across slices without a
mount. Both are backed by the same Ceph clusters.